# Lesson 73 · Evaluating with WER — the same 15% error rate, very different causes: diagnosing S/D/I sentence by sentence

**Goal**: break WER down into S / D / I and find out where the model is actually going wrong.

> Closing the loop with L67: one example each of insertion / deletion / substitution before we talk about rates. L74 continues with word-level error analysis.

🔗 Aurora connection: the `analyze_errors()` function you build here will later be folded into the `aurora/speech/` module (still at the planning stage, see `src/aurora/speech/__init__.py`), giving the `aurora.speech` fine-tuning loop a quantitative feedback signal.

← **Previous lesson** · [L72 · Fine-tuning Whisper](L72_whisper_finetune.ipynb)

> Last lesson covered **fine-tuning Whisper**: LoRA low-rank injection vs full fine-tuning, and adapting to a low-resource language in practice.
> This lesson explores **evaluating with WER**.

## Learning goals

By the end of this lesson you should be able to:

1. **Formula** — write WER = (S+D+I)/N and explain what each of the three error types — substitution (S), deletion (D), insertion (I) — tells you diagnostically;
2. **Implementation** — use a dynamic-programming edit alignment (DP back-trace) to read the S/D/I counts off a pair of word lists and output the exact WER;
3. **Analysis** — spot heavy-tail samples in the per-sentence WER distribution, and locate weak spots by domain (numbers / proper nouns / accents);
4. **Comparison** — explain the trade-offs between a library such as jiwer and a hand-written implementation (ready to use vs control over the details), and know when to use which.


## This lesson's story: two models with the same WER — which is easier to fix?

You fine-tuned Whisper on a Welsh dataset and got WER = 15%. A colleague also got WER = 15%. Whose model is better? The overall score isn't enough; you have to take the errors apart.

**Answer: you can't tell.** WER only tells you "how many words went wrong", not "how they went wrong".

```
Model A: WER=15%, all deletions (D_rate=15%)
           → channel noise? decoding stopping early? a language-model constraint might help.
Model B: WER=15%, all insertions (I_rate=15%)
           → model "hallucination"? noisy labels in the training data? needs a CTC constraint or confidence filtering.
```

The full WER formula:
```
WER = (S + D + I) / N
```
- S (Substitution): one word heard as another
- D (Deletion): a word that was left out
- I (Insertion): an extra word that wasn't there
- N: the total number of words in the reference text

**Diagnostic logic**:
| High error type | Likely cause | Direction for the fix |
|---|---|---|
| high S_rate | confusing similar sounds ("their"/"there") | more diverse data |
| high D_rate | channel noise / truncation | check VAD / padding |
| high I_rate | model "hallucination" | lower the beam width / add a penalty |

**This lesson's task**: implement `analyze_errors(hypotheses, references)` — output the overall WER and the three S/D/I rates for a batch.

### More on the story: why do different error types need different fixes?

| Error type | Diagnostic signal | Direction for the fix | The reason behind it |
|---------|--------|--------|--------|
| **High S_rate** (many substitutions) | "dog" heard as "bog", similar sounds confused | add **data diversity**, extend the vocabulary, add contrastive training | Substitutions mean the model is unstable in deciding "which word". Usually because: 1) the hard-to-tell word pair appears rarely in the training data, 2) the phoneme-level features aren't distinct enough (it needs more varied pronunciation contexts) |
| **High D_rate** (many deletions) | short words dropped, sentence endings cut off | check **VAD/padding**, add audio preprocessing, adjust the decoder's early-stopping threshold | Deletions mean some words were "never detected at all". Usually because: 1) voice activity detection (VAD) cut useful signal off as silence, 2) very short words (function words such as "a" or "of") were drowned out in low-energy stretches, 3) the decoder stopped outputting before it was confident enough |
| **High I_rate** (many insertions) | more output words than the audio's duration allows, hallucinated words | lower the **beam width** / raise the temperature / add a repetition penalty | Insertions mean the decoder outputs words where there's "no acoustic evidence". Usually because: 1) the beam search is too wide and explores too many low-confidence paths, 2) the language-model weight is too high, pushing out text that doesn't match the audio, 3) the training data has label noise (repeated words, redundant subtitles) |

**Key intuition**:
- **S** (substitution) is a classification problem → needs better **discrimination** → train on more diverse data
- **D** (deletion) is a detection problem → needs **more sensitive detection** → improve audio preprocessing, lower thresholds
- **I** (insertion) is a constraint problem → needs **tighter constraints** → shrink the search space, add penalty terms

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Breaking WER down: S / D / I

The WER formula:

```
WER = (S + D + I) / N
```

N is the total number of words in the reference, S = Substitution, D = Deletion, I = Insertion. The three error types are read off by **back-tracing the dynamic-programming (DP) edit distance**: during the back-trace, a diagonal step is a substitution/match, an upward step is an insertion (the hypothesis has one extra word), and a leftward step is a deletion (the reference has one extra word).

Typical distribution: in conversational speech S dominates (about 60-70%); with a strong accent D rises (dropped words); when the decoder is too aggressive I rises with hallucinated words. The three rates decide the direction of the fix.

### 1.1 An intuitive look at edit-distance DP: what the initialization and the DP matrix mean

Let me use the simplest possible example to explain the DP matrix's initialization and what its rows and columns mean.

**Setup**:
- horizontal axis j (columns): the words of the reference text ref
- vertical axis i (rows): the words of the hypothesis text hyp
- `dp[i][j]`: the minimum number of edit operations to turn the first i words of hyp into the first j words of ref

**What the initialization means geometrically**:

```
dp[i][0] = i  means "turn the first i words of hyp into nothing": you can only delete them one by one, i operations
dp[0][j] = j  means "turn nothing into the first j words of ref": you can only insert them one by one, j operations
```

A concrete example:
- hyp = ["A", "B"] (2 words)
- ref = [] (empty)
- To turn ["A", "B"] into [], all you can do is: delete "A", delete "B", 2 operations
- so `dp[2][0] = 2`

The other way round:
- hyp = [] (empty)
- ref = ["X", "Y"] (2 words)
- To turn nothing into ["X", "Y"], you can only insert one at a time: "insert X", "insert Y", 2 operations
- so `dp[0][2] = 2`

(Note: "insert/delete" here is the editing view of "turn hyp into ref". In WER bookkeeping, a word that ref has and hyp lacks is counted as D, and an extra hyp word as I — the back-trace in the next section uses the WER convention.)

**Matrix example** (hyp="AB", ref="XYZ"):

```
        ""  X   Y   Z
    ""  0   1   2   3      ← top row: hyp is empty, every ref word is missing (counted as D in WER)
    A   1   ?   ?   ?
    B   2   ?   ?   ?
    ↑
  first column: every hyp word is extra (counted as I in WER)
```

After initializing, fill along the diagonal. If hyp[i-1]==ref[j-1] (same word), then `dp[i][j] = dp[i-1][j-1]` (no new operation needed); otherwise take the minimum of the three neighbors + 1.

### 1.2 The back-trace: recovering the S/D/I operation sequence from the optimal edit distance

Once the DP matrix is built, start at `dp[H][R]` (bottom-right corner) and **walk back** to `dp[0][0]` (top-left corner). The direction of each step tells you what kind of error it is:

| Current position | Step to | Meaning | Operation |
|---------|---------|------|---------|
| (i, j) | (i-1, j-1) | hyp[i-1] ≠ ref[j-1] | substitution S: change the hyp word into the ref word |
| (i, j) | (i-1, j-1) | hyp[i-1] == ref[j-1] | match C: same word, no operation needed |
| (i, j) | (i-1, j)  | hyp has one extra word | insertion I: delete the hyp word |
| (i, j) | (i, j-1)  | ref has one extra word | deletion D: a ref word that isn't in hyp |

**Why do i / j correspond this way?**

Think of `dp[i][j]` as a coordinate table:
- `i` is how many hyp words have been consumed so far — "progress down the vertical axis"
- `j` is how many ref words have been consumed so far — "progress along the horizontal axis"
- so `dp[i][j]` isn't an abstract symbol, but the best cost "once i hyp words on the left and j ref words along the top have been used up"

That gives the three back-trace directions a natural meaning:
- up `(i, j) -> (i-1, j)`: one fewer hyp word, but no fewer ref words — this hyp word didn't match anything, so it's an I
- left `(i, j) -> (i, j-1)`: one fewer ref word, but no fewer hyp words — this ref word isn't in hyp, so it's a D
- diagonal `(i, j) -> (i-1, j-1)`: both sides step back one word — either a C (same word) or an S (different words, patched by a substitution)

**The core conditions**:

```
elif i > 0 and j > 0 and hyp[i-1] == ref[j-1]:
    ops.append('C')  # exact word match, step diagonally, and dp[i][j] == dp[i-1][j-1] (cost unchanged)

elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
    ops.append('S')  # words don't match but we step diagonally, so a substitution was used (cost +1)

elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
    ops.append('I')  # step up: this hyp word was skipped over in ref (insertion error)

else:  # the only option left is to step left
    ops.append('D')  # step left: this ref word was skipped over in hyp (deletion error)
```

**What if several conditions hold at once?**

Sometimes a cell satisfies more than one predecessor condition, which means there are several equally optimal back-trace paths. The `elif` chain simply picks one of those optimal paths — it isn't "changing the answer at random".

This affects the **labeling order** of `S / D / I`, but not the results that matter most:
- the minimum edit distance is unchanged
- the total `S + D + I` is unchanged
- so the final `WER = (S + D + I) / N` is unchanged

**Geometric picture** (grid): suppose hyp="AB", ref="XYZ"; the fully filled DP table looks like this:

```
       ref->" "   X    Y    Z
hyp    ""   0    1    2    3    (top row: hyp is empty, so X, Y, Z of ref are all missing = D)
 |      A   1    1    2    3
 v      B   2    2    2    3
```

The bottom-right corner is `dp[2][3] = 3`: the minimum edit distance is 3. Back-tracing from (2,3) in the order of the `elif`s above:

```
 (2,3)  B≠Z, but dp[2][3] == dp[1][2]+1 = 3     → diagonal, record S (B replaced by Z)
 (1,2)  A≠Y, but dp[1][2] == dp[0][1]+1 = 2     → diagonal, record S (A replaced by Y)
 (0,1)  i is already 0, so the only way is left  → record D (ref's X is missing from hyp)
 (0,0)
```

Read forward that's `D → S → S`: S=2, D=1, I=0, 3 steps in all, exactly `dp[2][3]`.

A counter-example while we're here: you could also go left 3 steps (3 D's) and then up 2 steps (2 I's) to reach (0,0). That path is legal too, but it costs 5, which isn't optimal. The reason the back-trace looks at the `dp` values instead of picking directions at will is precisely to walk only the path that costs 3.

In [ ]:
# Demo: build the full DP matrix and back-trace path for a small example by hand
# so you can see exactly how the matrix fills up and how the back-trace ends

def _edit_ops_with_matrix(hyp, ref):
    """Return (ops, dp matrix, back-trace path coordinates) for visualization"""
    H, R = len(hyp), len(ref)
    dp = [[0] * (R + 1) for _ in range(H + 1)]

    # initialize
    for i in range(H + 1):
        dp[i][0] = i
    for j in range(R + 1):
        dp[0][j] = j

    # fill
    for i in range(1, H + 1):
        for j in range(1, R + 1):
            if hyp[i - 1] == ref[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])

    # back-trace
    ops = []
    path = [(H, R)]  # start at the bottom-right corner
    i, j = H, R
    while i > 0 or j > 0:
        if i > 0 and j > 0 and hyp[i - 1] == ref[j - 1]:
            ops.append('C')
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            ops.append('S')
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append('I')
            i -= 1
        else:
            ops.append('D')
            j -= 1
        path.append((i, j))

    ops.reverse()
    path.reverse()
    return ops, dp, path


def _plot_dp_matrix(hyp, ref, dp, path, title):
    """Draw the DP matrix as a real grid and overlay the back-trace path."""
    H, R = len(hyp), len(ref)
    fig, ax = plt.subplots(figsize=(1.4 * (R + 1), 1.2 * (H + 1)))
    ax.imshow(np.array(dp), cmap='Blues', origin='upper')

    ax.set_xticks(range(R + 1))
    ax.set_yticks(range(H + 1))
    ax.set_xticklabels(['∅'] + ref)
    ax.set_yticklabels(['∅'] + hyp)
    ax.set_xlabel('ref progress j')
    ax.set_ylabel('hyp progress i')
    ax.set_title(title)

    for i in range(H + 1):
        for j in range(R + 1):
            ax.text(j, i, str(dp[i][j]), ha='center', va='center',
                    color='black', fontsize=11, fontweight='bold')

    xs = [j for i, j in path]
    ys = [i for i, j in path]
    ax.plot(xs, ys, color='crimson', lw=2.5, marker='o', markersize=6)

    for (i0, j0), (i1, j1) in zip(path[:-1], path[1:]):
        if i1 == i0 - 1 and j1 == j0 - 1:
            step = '↖'
        elif i1 == i0 - 1 and j1 == j0:
            step = '↑'
        else:
            step = '←'
        ax.text((j0 + j1) / 2, (i0 + i1) / 2, step,
                color='crimson', fontsize=14, ha='center', va='center', fontweight='bold')

    ax.set_aspect('equal')
    ax.grid(color='white', linewidth=1.5)
    plt.tight_layout()
    plt.show()


# Example 1: a simple case
hyp1 = ['cat', 'sat']
ref1 = ['the', 'cat', 'sat']
ops1, dp1, path1 = _edit_ops_with_matrix(hyp1, ref1)

print('=' * 60)
print("Example 1: hyp='cat sat', ref='the cat sat'")
print('=' * 60)
print('\nDP matrix (each cell is an edit distance):\n')
print('       ' + ' '.join(f"{w:>4}" for w in ['""'] + ref1))
for i in range(len(hyp1) + 1):
    row_label = '""' if i == 0 else hyp1[i - 1]
    print(f"{row_label:>4} {' '.join(f'{dp1[i][j]:>4}' for j in range(len(ref1) + 1))}")

print('\nBack-trace path and operation sequence:')
print(f"  path coordinates: {' -> '.join(f'({i},{j})' for i, j in path1)}")
print(f"  operation sequence: {' → '.join(ops1)}")
print(f"\nCounts: S={ops1.count('S')}  D={ops1.count('D')}  I={ops1.count('I')}  C={ops1.count('C')}")
print(f"WER = ({ops1.count('S')} + {ops1.count('D')} + {ops1.count('I')}) / {len(ref1)} = {(ops1.count('S') + ops1.count('D') + ops1.count('I')) / len(ref1):.3f}")

print('\nThis plot draws the same example\'s DP table as a real grid:')
_plot_dp_matrix(hyp1, ref1, dp1, path1, "Example 1: cat sat → the cat sat")

# Example 2: a case with a substitution error
hyp2 = ['the', 'dog', 'sat']
ref2 = ['the', 'cat', 'sat']
ops2, dp2, path2 = _edit_ops_with_matrix(hyp2, ref2)

print('\n' + '=' * 60)
print("Example 2: hyp='the dog sat', ref='the cat sat' (substitution error)")
print('=' * 60)
print('\nDP matrix:\n')
print('       ' + ' '.join(f"{w:>4}" for w in ['""'] + ref2))
for i in range(len(hyp2) + 1):
    row_label = '""' if i == 0 else hyp2[i - 1]
    print(f"{row_label:>4} {' '.join(f'{dp2[i][j]:>4}' for j in range(len(ref2) + 1))}")

print('\nBack-trace path and operation sequence:')
print(f"  path coordinates: {' -> '.join(f'({i},{j})' for i, j in path2)}")
print(f"  operation sequence: {' → '.join(ops2)}")
print(f"\nCounts: S={ops2.count('S')}  D={ops2.count('D')}  I={ops2.count('I')}  C={ops2.count('C')}")
print(f"WER = ({ops2.count('S')} + {ops2.count('D')} + {ops2.count('I')}) / {len(ref2)} = {(ops2.count('S') + ops2.count('D') + ops2.count('I')) / len(ref2):.3f}")


### 1.3 Important clarifications: the 'C' operation, string comparison, and special characters

**Q: How does the 'C' (Correct/match) operation relate to WER?**

In the code above you may have noticed the line `ops.append('C')`. To clarify:
- **C (Correct, match)**: the two words are identical — recognized correctly, no operation needed
- **WER uses only S/D/I**: `WER = (S + D + I) / N`, **C doesn't count towards WER**
- **What C is for**: checking the alignment. The number of C's in the operation sequence should ≈ the number of words recognized exactly right

Example: hyp="the cat sat", ref="the cat sat"
- operation sequence: ['C', 'C', 'C']
- S=0, D=0, I=0, WER=0 (perfectly correct)

---

**Q: Is string comparison case-sensitive? What about symbols? How are numbers tokenized?**

This course's conventions:
1. **Case-sensitive** ("The" and "the" are different) — if your application should ignore case, just lowercase everything before computing WER: `hyp = [w.lower() for w in hyp_raw]`
2. **Symbols and numbers are kept as they are** — they're part of the word and compared with exact `==`
3. **The "inconsistent spelling" problem with numbers** (Samuel's question 5):
   - ref: ["one", "nine", "nine", "seven"] (four digits, each its own word)
   - hyp: ["1997"] (the number as one word)
   - or hyp: ["one-nine-nine-seven"] (the four digits glued into one word)

   WER differs across these three cases. **The preprocessing choice**:
   - keep the tokenization as is → WER reflects the cost of the "spelling difference"
   - map all numbers to a "<NUM>" tag → WER only looks at "whether the position is right" and ignores the number format

   This course has no single standard; in your own project, **state clearly in the documentation which scheme you use**.

In [ ]:
# Demo: align a pair of sentences by hand and look at the S/D/I operations
def _edit_ops(hyp, ref):
    H, R = len(hyp), len(ref)
    dp = [[0]*(R+1) for _ in range(H+1)]
    for i in range(H+1): dp[i][0] = i
    for j in range(R+1): dp[0][j] = j
    for i in range(1, H+1):
        for j in range(1, R+1):
            if hyp[i-1] == ref[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    ops = []
    i, j = H, R
    while i > 0 or j > 0:
        if i > 0 and j > 0 and hyp[i-1] == ref[j-1]:
            ops.append('C'); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append('S'); i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append('I'); i -= 1   # hyp has one extra word
        else:
            ops.append('D'); j -= 1   # ref has one extra word
    ops.reverse()
    return ops

hyp = ['the', 'cat', 'sat']
ref = ['the', 'big', 'cat', 'sat', 'down']
ops = _edit_ops(hyp, ref)
counts = {op: ops.count(op) for op in 'SDIC'}
print('Alignment operation sequence:', ops)
print(f"S={counts['S']}  D={counts['D']}  I={counts['I']}  C={counts['C']}")
print(f"WER = ({counts['S']}+{counts['D']}+{counts['I']}) / {len(ref)} "
      f"= {(counts['S']+counts['D']+counts['I'])/len(ref):.2f}")

## 2. The per-sentence WER distribution

Look sentence by sentence and WER has a **heavy-tail distribution**: most sentences have WER=0 (recognized perfectly), while a few have a very high WER, even above 100%. Going over 100% happens when the number of inserted words I exceeds the number of reference words N:

```
WER_i > 1.0  ↔  I_i > N_i  (more hallucinated words than real ones)
```

A few extreme sentences pull the mean up, so the **median** better reflects how most sentences do. Pay special attention to samples with WER > 50% — they usually correspond to noisy environments, non-native accents, domain-specific proper nouns or very long sentences, and they're the first candidates for data augmentation during fine-tuning.

### 2.1 Concrete examples of WER > 100%: when, why, and what it means

WER > 1.0 sounds extreme, but it really does happen. Let's look at a few concrete examples to understand it.

In [ ]:
# Concrete examples of WER > 100%
print("=" * 70)
print("Demonstrating cases with WER > 100%")
print("=" * 70)

# Example 1: the model "hallucinates" and inserts lots of words
hyp_halluc = ['the', 'cat', 'is', 'sitting', 'on', 'the', 'mat', 'which', 'is', 'very', 'comfortable', 'and']
ref_short = ['cat']
ops_h = _edit_ops(hyp_halluc, ref_short)
S_h = ops_h.count('S')
D_h = ops_h.count('D')
I_h = ops_h.count('I')
N_h = len(ref_short)
wer_h = (S_h + D_h + I_h) / N_h

print("\nExample 1: the model over-generates (hallucination)")
print(f"  ref: {ref_short}  (1 word)")
print(f"  hyp: {hyp_halluc}  (12 words)")
print(f"  operations: {ops_h}")
print(f"  S={S_h}, D={D_h}, I={I_h}, N={N_h}")
print(f"  WER = ({S_h}+{D_h}+{I_h})/{N_h} = {wer_h:.1f} = {wer_h*100:.0f}%  ← WER > 1!")
print(f"  Reading: the decoder inserted lots of extra words, 12 times as long as the original. This usually means:")
print(f"        • the language model is over-driven (temperature too low)")
print(f"        • the training data has a lot of repeated, redundant labels")
print(f"        • CTC merging failed, so words were repeated several times")

# Example 2: a medium-length sentence with a high hallucination rate
hyp_med = ['hello', 'world', 'is', 'a', 'test', 'and', 'also', 'a', 'demo', 'sentence']
ref_med = ['hello', 'world']
ops_m = _edit_ops(hyp_med, ref_med)
S_m = ops_m.count('S')
D_m = ops_m.count('D')
I_m = ops_m.count('I')
N_m = len(ref_med)
wer_m = (S_m + D_m + I_m) / N_m

print("\nExample 2: a medium-length sentence with a high hallucination rate")
print(f"  ref: {ref_med}  (2 words)")
print(f"  hyp: {hyp_med}  (10 words)")
print(f"  operations: {ops_m}")
print(f"  S={S_m}, D={D_m}, I={I_m}, N={N_m}")
print(f"  WER = ({S_m}+{D_m}+{I_m})/{N_m} = {wer_m:.1f} = {wer_m*100:.0f}%")
print(f"  Reading: the model recognized the first two words, but everything after is hallucinated. Directions for the fix:")
print(f"        • lower the beam width (narrow the search)")
print(f"        • add a repetition penalty (penalty for repeated tokens)")
print(f"        • check whether the decoder is losing BOS/EOS tokens")

# Example 3: why deletions + substitutions can't push WER above 1.0 (but insertions can)
print("\nExample 3: the upper bound for deletions and substitutions")
print("  With only deletions (D) and substitutions (S), what's the worst case?")
print("    • most deletions: delete all N reference words → D=N, contributing N/N=1.0")
print("    • most substitutions: substitute all N reference words → S=N, contributing N/N=1.0")
print("    • deletions + substitutions: each reference word is deleted or substituted at most once, so (D+S)/N ≤ N/N = 1.0")
print("  What's different about insertions:")
print("    • any number of words can be inserted (not limited by N) → I can be huge → WER has no upper bound")
print("  That's why WER > 1.0 is always caused by **insertion errors**.")

In [ ]:
# Demo: simulate a batch of WERs and look at the distribution and its long tail
np.random.seed(42)
wers = np.concatenate([
    np.random.beta(1, 8, 80) * 0.4,   # low-error bulk
    np.random.uniform(0.5, 1.5, 15),   # high-error long tail
    np.random.uniform(1.0, 2.0, 5),    # WER > 1 (too many insertions)
])

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(wers, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(wers), color='red', linestyle='--',
                label=f'mean {np.mean(wers):.2f}')
axes[0].axvline(np.median(wers), color='orange', linestyle='--',
                label=f'median {np.median(wers):.2f}')
axes[0].set_xlabel('WER'); axes[0].set_ylabel('sentences')
axes[0].set_title('Per-sentence WER distribution (simulated)')
axes[0].legend()

axes[1].hist(wers, bins=30, color='steelblue', edgecolor='white',
             cumulative=True, density=True)
axes[1].axhline(0.8, color='green', linestyle=':', label='80th percentile')
axes[1].set_xlabel('WER'); axes[1].set_ylabel('cumulative fraction')
axes[1].set_title('Cumulative distribution (CDF)')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Sentences with WER > 50%: {(wers > 0.5).sum()} / {len(wers)}')
print(f'Sentences with WER > 100%: {(wers > 1.0).sum()} / {len(wers)}')

## 3. Analysis by domain: numbers / proper nouns / accents

The overall WER is a weighted average of the subsets, and the error types differ clearly from domain to domain:

- **Numbers**: inconsistent spelling ("nineteen ninety-seven" vs "1997") inflates I/S — semantically right, but the WER comes out high
- **Proper nouns / jargon**: poor coverage in training, so S (substitution) dominates
- **Accents / dialects**: phonemes aren't recognized, so words get dropped and D (deletion) rises

```
Diagnostic flow:
  bucket by domain → compute (S_rate, D_rate, I_rate) separately for each bucket
  → differences in the error-type distribution → point to a specific fix
```

Proper nouns → extend the vocabulary / prompts; accents → data augmentation; numbers → post-processing rules.

### 3.1 Going deeper: number recognition and "fake" WER inflation

Section 3 mentioned the "inconsistent spelling" problem with numbers. Let me expand on it with concrete examples.

In [ ]:
# A concrete example of number recognition inflating WER

print("=" * 70)
print("Number recognition: inflated WER, or a real error?")
print("=" * 70)

# Scenario: the audio says "the bridge opened in 2024" — the speaker said the number "2024"

# Scheme A: tokenize strictly by pronunciation — "two zero two four"
ref_a = ["bridge", "opened", "in", "two", "zero", "two", "four"]
hyp_a = ["bridge", "opened", "in", "2", "0", "2", "4"]  # the model outputs digits, also 7 words
ops_a = _edit_ops(hyp_a, ref_a)
S_a = ops_a.count('S'); D_a = ops_a.count('D'); I_a = ops_a.count('I')
wer_a = (S_a + D_a + I_a) / len(ref_a)

print("\nScheme A: literal matching ('two' ≠ '2', counted as a substitution)")
print(f"  ref: {ref_a}")
print(f"  hyp: {hyp_a}")
print(f"  operations: {ops_a}")
print(f"  WER = ({S_a}+{D_a}+{I_a})/{len(ref_a)} = {wer_a:.3f}")
print(f"  ❌ Problem: semantically the model got it right (2024), but the WER is high (4/7≈57%)")

# Scheme B: normalize to a tag — every number becomes <NUM>
ref_b = ["bridge", "opened", "in", "<NUM>"]
hyp_b = ["bridge", "opened", "in", "<NUM>"]
ops_b = _edit_ops(hyp_b, ref_b)
S_b = ops_b.count('S'); D_b = ops_b.count('D'); I_b = ops_b.count('I')
wer_b = (S_b + D_b + I_b) / len(ref_b)

print("\nScheme B: normalize in preprocessing (numbers → <NUM> tag)")
print(f"  ref: {ref_b}  (with the regular expression r'\\d+' → '<NUM>')")
print(f"  hyp: {hyp_b}")
print(f"  operations: {ops_b}")
print(f"  WER = ({S_b}+{D_b}+{I_b})/{len(ref_b)} = {wer_b:.3f}")
print(f"  YES: WER=0, an accurate picture of the model's ability to place the number")

# Another example: how a year is written
print("\nAnother example: the year '1997'")
hyp_1997 = ["nineteen", "ninety", "seven"]
ref_1997 = ["one", "thousand", "nine", "hundred", "ninety", "seven"]
ops_1997 = _edit_ops(hyp_1997, ref_1997)
S_1997 = ops_1997.count('S'); D_1997 = ops_1997.count('D'); I_1997 = ops_1997.count('I')
wer_1997 = (S_1997 + D_1997 + I_1997) / len(ref_1997)

print(f"  ref: {ref_1997}  (in full: one thousand nine hundred ninety seven)")
print(f"  hyp: {hyp_1997}  (short form: nineteen ninety seven)")
print(f"  operations: {ops_1997}")
print(f"  WER = {wer_1997:.3f}  (the model used the short form, which is more natural, yet the WER is still high)")
print("\nWarning:")
print("  * When recognizing numbers/years, WER is easily inflated (right meaning, different format)")
print("  * In your project, state clearly in the documentation:")
print("    - whether WER is computed on the literal text (to see the cost of format differences)")
print("    - or after normalization in preprocessing (caring only about position and category)")
print("  * Consider setting up separate domain diagnostics for numbers, dates, proper nouns, etc.")
print("    so they accurately reflect the true error rate of these sub-problems")

In [ ]:
# Demo: bar chart comparing S/D/I across three domains
domains = ['General', 'With numbers', 'With proper nouns']
stats = {
    'General':           {'S': 0.07, 'D': 0.03, 'I': 0.02},
    'With numbers':      {'S': 0.04, 'D': 0.02, 'I': 0.12},  # many hallucinated words, high I
    'With proper nouns': {'S': 0.18, 'D': 0.05, 'I': 0.03},  # many substitutions, high S
}

x = np.arange(len(domains))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 4))
for i, (op, color, label) in enumerate([
    ('S', '#E07B54', 'S substitution'),
    ('D', '#5B8DB8', 'D deletion'),
    ('I', '#6DAC6D', 'I insertion'),
]):
    vals = [stats[d][op] for d in domains]
    ax.bar(x + i*width, vals, width, label=label, color=color)
ax.set_xticks(x + width)
ax.set_xticklabels(domains)
ax.set_ylabel('error rate')
ax.set_title('S / D / I error rates by domain')
ax.legend()
plt.tight_layout()
plt.show()

## 4. ✏️ Implement `analyze_errors(hypotheses, references)`

**Signature**:
```python
def analyze_errors(hypotheses: list[list[str]], references: list[list[str]]) -> dict:
    # hypotheses: list of word lists (model output, already tokenized)
    # references: list of word lists (reference text, already tokenized)
    # returns: {"wer", "S_rate", "D_rate", "I_rate", "worst_examples"}, 5 keys in all
```

**A 3-step plan**:

| Step | What to do | Tool |
|---|---|---|
| 1 | For each `(hyp, ref)` pair, count (S, D, I): write the DP + back-trace as in `_edit_ops` above, or run `ops.count('S')` etc. on the operation sequence it returns | the `edit_counts` TODO |
| 2 | Accumulate S/D/I and the total number of reference words N over all sentences | `sum()` |
| 3 | Compute the overall WER=(S+D+I)/N, S_rate=S/N, D_rate=D/N, I_rate=I/N | watch the edge case N=0 |

**Acceptance criteria**:
- `hyp=["hello"], ref=["hello", "world"]` → D=1, S=0, I=0, N=2, WER=0.5
- `hyp=["a","b","c","x"], ref=["a","b","c"]` → I=1, N=3, WER≈0.333
- the returned dict has 5 keys (wer / S_rate / D_rate / I_rate / worst_examples), and WER is a float

**What `worst_examples` should look like**:

`worst_examples` is a list holding up to 5 samples with the highest WER. Each sample is a **tuple** or a **dict**, like this:

```python
# Format 1: tuple (recommended for a compact implementation)
worst_examples = [
    (idx, wer_value),      # idx is the sample's position in the input list, wer_value is that sentence's WER
    (idx, wer_value),
    ...
]

# Format 2: dict (if you need more information)
worst_examples = [
    {"idx": idx, "wer": wer_value, "hyp": hyp_words, "ref": ref_words},
    ...
]
```

**A concrete example**:
```python
result = analyze_errors(
    hypotheses=[
        ["hello", "world"],      # idx=0, perfectly right, WER=0
        ["the", "cat"],          # idx=1, a word missing, WER=0.33
        ["a"],                   # idx=2, high error, WER=0.75
    ],
    references=[
        ["hello", "world"],
        ["the", "big", "cat"],
        ["a", "b", "c", "d"],
    ]
)

# return value (example, format 1): 4 deletions out of N=9 reference words
result = {
    'wer': 0.44,
    'S_rate': 0.0,
    'D_rate': 0.44,
    'I_rate': 0.0,
    'worst_examples': [
        (2, 0.75),  # idx=2 has the highest WER
        (1, 0.33),  # idx=1 has the second-highest WER
        (0, 0.00),
    ]
}
```

In [ ]:
def analyze_errors(hypotheses, references):
    # hypotheses: list[list[str]]  model recognition output (a list of word lists)
    # references:  list[list[str]]  reference text (a list of word lists)
    # returns: dict with wer / S_rate / D_rate / I_rate / worst_examples

    def edit_counts(hyp, ref):
        # ✏️ TODO: build the dp matrix with dynamic programming, then back-trace and return (S, D, I)
        raise NotImplementedError("TODO: implement the dynamic-programming edit alignment, return the (S, D, I) triple")

    total_S = total_D = total_I = total_N = 0
    per_wer = []
    for hyp, ref in zip(hypotheses, references):
        # ✏️ TODO: call edit_counts, accumulate S/D/I/N, record the per-sentence WER
        raise NotImplementedError("TODO: call edit_counts, accumulate S/D/I/N, compute the per-sentence WER")

    # ✏️ TODO: from per_wer, pick out the 5 samples with the highest WER (with their indices)
    worst = []

    return {
        'wer': (total_S + total_D + total_I) / max(total_N, 1),
        'S_rate': total_S / max(total_N, 1),
        'D_rate': total_D / max(total_N, 1),
        'I_rate': total_I / max(total_N, 1),
        'worst_examples': worst,
    }


In [ ]:
# Check: verify against known answers
# hyp[0] is missing a word (D=1, N=3); hyp[1] has an extra word (I=1, N=2)
# Total: S=0, D=1, I=1, N=5  →  WER = 2/5 = 0.40
# Exact expectations: S_rate=0/5=0.0, D_rate=1/5=0.2, I_rate=1/5=0.2
hyps = [['the', 'cat'], ['hello', 'world', 'today']]
refs = [['the', 'big', 'cat'], ['hello', 'world']]

try:
    result = analyze_errors(hyps, refs)
except (NotImplementedError, TypeError) as _e:
    print(f"⬜ analyze_errors not implemented: {_e}")
    result = None

if result is not None:
    assert isinstance(result, dict), 'The return value should be a dict'
    assert {'wer','S_rate','D_rate','I_rate','worst_examples'} <= set(result.keys()), 'Missing a required key'

    # Exact WER assertion (Fix C2)
    assert abs(result['wer'] - 0.40) < 1e-9, (
        f"Expected WER 0.400, got {result['wer']:.6f}; "
        f"check that the S/D/I counts are right (S=0,D=1,I=1,N=5)"
    )
    # Exact assertions for each rate
    assert abs(result['S_rate'] - 0.0) < 1e-9, f"Expected S_rate 0.0, got {result['S_rate']:.6f}"
    assert abs(result['D_rate'] - 0.2) < 1e-9, f"Expected D_rate 0.200, got {result['D_rate']:.6f}"
    assert abs(result['I_rate'] - 0.2) < 1e-9, f"Expected I_rate 0.200, got {result['I_rate']:.6f}"

    rate_sum = result['S_rate'] + result['D_rate'] + result['I_rate']
    assert abs(rate_sum - result['wer']) < 1e-9, (
        f'The S+D+I rates should add up to WER, got {rate_sum:.4f} vs {result["wer"]:.4f}'
    )
    assert result['wer'] >= 0.0, f"WER can't be negative, got {result['wer']:.3f}"
    assert len(result['worst_examples']) <= 5, 'worst_examples holds at most 5 entries'

    print('✅ analyze_errors passed every check')
    print(f"  WER     = {result['wer']:.3f}  ✓ exactly 0.400")
    print(f"  S_rate  = {result['S_rate']:.3f}  D_rate = {result['D_rate']:.3f}  I_rate = {result['I_rate']:.3f}")
    print(f"  worst_examples: {result['worst_examples']}")


## Going further: comparing with the jiwer library

The hand-written `analyze_errors()` is the heart of this lesson (understanding how the DP back-trace works). In production, the [jiwer](https://github.com/jitsi/jiwer) library is a common quick reference:

```python
# pip install jiwer  (optional, doesn't affect this lesson's grading)
import jiwer

reference = "the big cat"
hypothesis = "the cat"

out = jiwer.process_words(reference, hypothesis)   # jiwer 3.x+; the old compute_measures is deprecated and removed in jiwer 4
# output object: out.wer ≈ 0.333, out.deletions = 1, out.mer, out.wil, ...
# note: for a single sentence jiwer uses N=3, WER=D/N=1/3≈0.333
```

**The trade-offs**:

| Dimension | Hand-written | jiwer |
|------|----------|-------|
| Control | complete (alignment rules can be customized) | limited (fixed implementation) |
| Speed | O(H·R) per pair | same order, but C-accelerated |
| Interpretability | reads S/D/I back directly | you parse the output fields |
| Research use | ✅ recommended | for quick checks |

> The Aurora project follows the "No black boxes" principle — the hand-written `analyze_errors()` is the core deliverable; jiwer is only a reference.


## 5. Parameter experiment: WER histograms and a baseline comparison

**Experiment parameters**:
- `bins`: the number of histogram bins — 10 shows the rough shape, 50 shows the details of the long tail
- `finetuned_wers`: the fine-tuned model's per-sentence WER list (from walking per_wer in `analyze_errors`)
- `baseline_wers`: the per-sentence WER list of an un-fine-tuned Whisper-small on the same test set

**What to expect**:
- after fine-tuning, the whole distribution shifts left, with more sentences piling up in the low-WER range (0–0.1)
- fewer extreme samples with WER > 1.0, showing that hallucinated words are being suppressed
- if the mean drops less than the median, there's still a batch of especially stubborn samples in the long tail, and you'll need data aimed specifically at them

In [ ]:
# Parameter experiment: WER distribution before vs after fine-tuning (swap in real evaluation results when you use it)
np.random.seed(0)

baseline_wers = np.concatenate([
    np.random.beta(2, 5, 70) * 0.8,
    np.random.uniform(0.6, 2.0, 30),
])
finetuned_wers = np.concatenate([
    np.random.beta(1, 10, 85) * 0.4,
    np.random.uniform(0.4, 1.2, 15),
])

bins = 30  # ✏️ change bins to see coarser or finer detail

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(baseline_wers, bins=bins, alpha=0.6, color='#E07B54',
        label=f'Whisper-small baseline  mean={np.mean(baseline_wers):.2f}  '
              f'median={np.median(baseline_wers):.2f}')
ax.hist(finetuned_wers, bins=bins, alpha=0.6, color='#5B8DB8',
        label=f'fine-tuned              mean={np.mean(finetuned_wers):.2f}  '
              f'median={np.median(finetuned_wers):.2f}')
ax.axvline(1.0, color='black', linestyle=':', linewidth=1,
           label='WER = 1.0 boundary (more hallucinated words than reference words)')
ax.set_xlabel('per-sentence WER')
ax.set_ylabel('sentences')
ax.set_title('WER distribution before vs after fine-tuning')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Baseline   sentences with WER>1.0: {(baseline_wers > 1.0).sum()}')
print(f'Fine-tuned sentences with WER>1.0: {(finetuned_wers > 1.0).sum()}')

## Wrap-up

This lesson implemented `analyze_errors()`, which outputs the overall WER and the three rates `S_rate / D_rate / I_rate`, and pinpoints the 5 sentences with the highest WER in `worst_examples`. The function will later be folded into the `aurora/speech/` module (still at the planning stage) as the quantitative feedback signal for the `aurora.speech` fine-tuning loop. The next lesson (L74) digs into S/D/I error patterns: building a substitution confusion matrix, counting the most frequently substituted / deleted / inserted words, and turning WER from a single number into an actionable list of word errors.

## ✏️ Whiteboard challenge: diagnosing WER by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: ref="I love deep learning", hyp="I love learning". WER=? S=? D=? I=?
(S=0, D=1 ("deep" deleted), I=0, N=4; WER=1/4=0.25)

**Question 2**: ref="A B C", hyp="X A B C". WER=? What kind of error is it?
(I=1, S=0, D=0, N=3; WER=1/3≈0.333; an insertion error — the model "hallucinated" "X")

**Question 3**: when WER is lowest, is the model necessarily the best? Give a counterexample.
(Not necessarily: ref="weather is good", hyp="weather is very good" → WER=1/3; but "very" is a correct insertion (an error in the reference, or a dialect form). WER measures the difference from the reference text, not semantic correctness)

**Question 4**: if a sentence has N=10 words, what's the largest possible WER?
(Unbounded: as long as hyp inserts enough words, WER=(0+0+I)/10 can grow without limit; deletions + substitutions top out at (D+S)/10≤(10+0)/10=1.0; so WER has no upper bound)

**Question 5**: if you want to diagnose "the model does badly on sentences containing numbers", how would you analyze it?
(Analyze by subset: filter the (hyp, ref) pairs that contain numbers and compute their S/D/I distribution separately; see whether S_rate or I_rate is high — numbers are usually substitution errors, because they can be written many ways: "nineteen ninety-seven" vs "1997")

Once you've derived them, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np
import re

# Q1: a classic WER by hand
hyp_q1 = "I love learning".split()
ref_q1 = "I love deep learning".split()
ops_q1 = _edit_ops(hyp_q1, ref_q1)
S1 = ops_q1.count('S'); D1 = ops_q1.count('D'); I1 = ops_q1.count('I')
N1 = len(ref_q1)
wer_q1 = (S1 + D1 + I1) / N1
assert D1 == 1 and S1 == 0 and I1 == 0, f"Expected S=0,D=1,I=0, got S={S1},D={D1},I={I1}"
assert abs(wer_q1 - 0.25) < 1e-6, f"WER should be 0.25, got {wer_q1:.4f}"
print(f"Q1 ✅  'I love learning' vs 'I love deep learning': S={S1},D={D1},I={I1},N={N1},WER={wer_q1:.2f}")

# Q2: an insertion error
hyp_q2 = "X A B C".split()
ref_q2 = "A B C".split()
ops_q2 = _edit_ops(hyp_q2, ref_q2)
S2 = ops_q2.count('S'); D2 = ops_q2.count('D'); I2 = ops_q2.count('I')
N2 = len(ref_q2)
wer_q2 = (S2 + D2 + I2) / N2
assert I2 == 1 and S2 == 0 and D2 == 0, f"Expected I=1,S=0,D=0, got S={S2},D={D2},I={I2}"
print(f"Q2 ✅  'X A B C' vs 'A B C': S={S2},D={D2},I={I2},N={N2},WER={wer_q2:.3f} (insertion error)")

# Q3: analyze_errors acceptance criteria
try:
    # criterion 1: deletion
    r1 = analyze_errors([["hello"]], [["hello", "world"]])
    assert abs(r1["wer"] - 0.5) < 1e-6, f"WER should be 0.5, got {r1['wer']:.4f}"
    # criterion 2: insertion
    r2 = analyze_errors([["a","b","c","x"]], [["a","b","c"]])
    assert abs(r2["wer"] - 1/3) < 1e-6, f"WER should be 0.333, got {r2['wer']:.4f}"
    # criterion 3: S/D/I breakdown
    assert set(r1.keys()) >= {"wer","S_rate","D_rate","I_rate"}, f"Missing keys: {r1.keys()}"
    print(f"Q3 ✅  analyze_errors acceptance: deletion WER={r1['wer']:.2f}✓, insertion WER={r2['wer']:.3f}✓, 4 keys✓")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  analyze_errors: 3-step implementation (edit_ops → accumulate → compute rates), check again once implemented")

# Q4: the WER upper bound
wer_extreme = (0 + 0 + 100) / 10  # 100 words inserted
print(f"Q4 ✅  WER is unbounded: with N=10, inserting 100 words → WER={wer_extreme:.1f} (>1.0); deletions + substitutions max out at 1.0")

# Q5: a subset-analysis demo
def has_digits(words):
    return any(re.search(r'\d', w) for w in words)

sample_refs = [["today", "2024"], ["nice", "weather"], ["I", "live", "at", "No.1997"]]
sample_hyps = [["today", "two-thousand"], ["nice", "weather"], ["I", "live", "at", "No.nineteen-ninety-seven"]]
digit_pairs = [(h, r) for h, r in zip(sample_hyps, sample_refs) if has_digits(r)]
print(f"Q5 ✅  subset with numbers: {len(digit_pairs)} pairs (picked out of {len(sample_refs)}) → compute S/D/I separately to diagnose number recognition")
print("\n🎉 WER diagnosis whiteboard challenge passed!")


In [ ]:
# ✏️ Self-review for this lesson
l73_review = {
    "wer_formula_3_types":    None,  # Remember WER=(S+D+I)/N and can tell apart what S/D/I each diagnose? True/False
    "wer_no_upper_bound":     None,  # Understand that WER has no upper bound (many insertions can make WER>>1)? True/False
    "analyze_errors_impl":    None,  # analyze_errors() implemented correctly, 3 acceptance criteria pass? True/False
    "subset_diagnosis":       None,  # Understand how to analyze the WER distribution by domain/subset to find the model's weak spots? True/False
    "whiteboard_passed":      None,  # All 5 whiteboard derivations done? True/False
}

unfilled = [k for k, v in l73_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l73_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L73 all clear! On to L74: ASR error analysis')

---

→ **Next lesson** · [L74 · ASR error analysis](L74_asr_error_analysis.ipynb)

> Next lesson covers **ASR error analysis**: substitution / deletion / insertion patterns, from a WER number to what to fix.